#Hw2.3.ipynb

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("/content/2017_jun_final - 2017_jun_final.csv")

df.columns = df.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

df = df[['Посада', 'Мова програмування', 'Загальний досвід роботи', 'Досвід роботи на поточному місці', 'Зарплата на місяць', 'Дата заповнення', 'Спеціалізація']]
df.columns = ['name', 'author', 'user_rating', 'reviews', 'price', 'year', 'genre']

df['year'] = pd.to_datetime(df['year'], format='%d/%m/%Y %H:%M:%S').dt.year

def convert_experience_to_numeric(experience_str):
    if pd.isna(experience_str):
        return None
    experience_str = str(experience_str).lower()
    if 'менше 3 місяців' in experience_str:
        return 0.25
    elif '0.5' == experience_str:
        return 0.5
    elif '10 і більше років' in experience_str:
        return 10.0
    elif 'рік' in experience_str or 'роки' in experience_str or 'років' in experience_str:
        try:
            import re
            match = re.search(r'(\d+\.?\d*)', experience_str)
            if match:
                return float(match.group(1))
        except ValueError:
            pass
    try:
        return float(experience_str)
    except ValueError:
        return None

df['user_rating'] = df['user_rating'].apply(convert_experience_to_numeric)
df['reviews'] = df['reviews'].apply(convert_experience_to_numeric)

df.head()
df.shape

df.isna().sum()

df['genre'].unique()

df['price'].plot(kind='hist')

df['price'].max()
df['price'].min()
df['price'].mean()
df['price'].median()

df['user_rating'].max()

(df['user_rating'] == df['user_rating'].max()).sum()

if not df['reviews'].dropna().empty:
    df.loc[df['reviews'].idxmax()]
else:
    print("Warning: 'reviews' column might contain no valid numeric data after conversion for idxmax().")



df_2017 = df[df['year'] == 2017]

if not df_2017.empty and not df_2017['price'].dropna().empty:
    df_2017.loc[df_2017['price'].idxmax()]
else:
    print("Warning: df_2017 is empty or 'price' column in df_2017 has no valid numeric data for idxmax().")


len(df[(df['genre'] == 'Fiction') & (df['year'] == 2017)])

len(df[(df['user_rating'] == df['user_rating'].max()) & (df['year'] == 2017)])

df_2017_under8 = df[(df['year'] == 2017) & (df['price'] < 8)].sort_values('price')
if not df_2017_under8.empty:
    df_2017_under8.tail(1)
else:
    print("Warning: df_2017_under8 is empty.")

genre_prices = df.groupby('genre').agg({'price': ['max', 'min']})

author_counts = df.groupby('author').agg({'name': 'count'})
author_counts

author_counts.shape

author_counts.sort_values('name', ascending=False).head(1)

df.groupby('author').agg({'user_rating': 'mean'}).sort_values('user_rating').head(1)

author_ratings = df.groupby('author').agg({'user_rating': 'mean'})

merged = pd.concat([author_counts, author_ratings], axis=1)

merged.sort_values(['name', 'user_rating']).head(1)